In [ ]:
# Step 1: Install dependencies & set Hugging Face token
%pip install -q "datasets>=2.19.0" "huggingface_hub>=0.24"
import os
import getpass

# 直接设置Hugging Face token，跳过登录界面
hf_token = getpass.getpass("Paste your Hugging Face token: ")
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token

print("Hugging Face token set successfully!")

In [ ]:
# Step 2: Load FLARE-CAUSAL20-SC test set and preprocess
from datasets import load_dataset, Dataset

# FLARE-CAUSAL20-SC dataset
print("加载 FLARE-CAUSAL20-SC 数据集...")
ds_raw = load_dataset("ChanceFocus/flare-causal20-sc", split="test")
print(f"成功加载! 样本数: {len(ds_raw)}, 列: {ds_raw.column_names}")

# 定义标签
LABELS = ["noise", "causal"]

# 检查是否有标注
def _map_row(x):
    text = x.get("text") or x.get("sentence") or ""
    answer = x.get("answer", "")
    gold = x.get("gold", -1)
    choices = x.get("choices", LABELS)
    
    # 确定真实标签
    if gold >= 0 and gold < len(choices):
        true_label = choices[gold]
        has_label = True
    else:
        true_label = "unknown"
        has_label = False
    
    return {
        "text": text,
        "answer": answer,
        "gold": gold,
        "choices": choices,
        "label": true_label,
        "has_label": has_label,
        "labels": LABELS
    }

ds = Dataset.from_list([_map_row(r) for r in ds_raw])
bad = [i for i, r in enumerate(ds) if not r["text"]]
print(f"空文本样本数: {len(bad)}")

# 检查是否有标注
has_label_count = sum(1 for r in ds if r["has_label"])
print(f"有标注的样本数: {has_label_count}/{len(ds)}")

if has_label_count == 0:
    print("\n⚠️ 警告: 测试集没有标注，将只能进行推理测试，无法计算准确率")

# 显示第一个样本作为验证
print("\n第一个样本示例:")
print(f"文本: {ds[0]['text'][:200]}...")
print(f"标签: {ds[0]['label']}")
print(f"是否有标注: {ds[0]['has_label']}")

In [ ]:
# Step 3: Install dependencies, configure OpenAI, and record experiment metadata
%pip install -q "openai==1.40.2" "httpx==0.27.2" "httpcore==1.0.5" \
               "pandas>=2.2.2" "tqdm>=4.66.4" "requests>=2.31.0"

import os, getpass, json, time, platform
from importlib.metadata import version, PackageNotFoundError
import requests

# o3-mini配置
MODEL = "o3-mini"
BASE_URL = "https://api.openai.com/v1"

api_key = os.getenv("OPENAI_API_KEY") or os.getenv("API_KEY")
if not api_key:
    api_key = getpass.getpass("Paste your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# 文件命名
dataset_name = "flare_causal20_sc"
run_tag = f"{dataset_name}_{MODEL.replace('-', '_')}"
save_dir = os.getcwd()
pred_path = os.path.join(save_dir, f"{run_tag}_predictions.csv")
meta_path = os.path.join(save_dir, f"{run_tag}_metadata.json")

def ver(pkg: str) -> str:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not-installed"

meta = {
    "dataset": "ChanceFocus/flare-causal20-sc",
    "split": "test",
    "labels": LABELS,
    "model": MODEL,
    "has_labels": has_label_count > 0,
    "labeled_samples": has_label_count,
    "openai_sdk": ver("openai"),
    "httpx": ver("httpx"),
    "httpcore": ver("httpcore"),
    "datasets_version": ver("datasets"),
    "pandas": ver("pandas"),
    "tqdm": ver("tqdm"),
    "time_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": platform.python_version(),
    "base_url": BASE_URL,
    "note": "o3-mini evaluation on FLARE-CAUSAL20-SC (Causal Classification)"
}

os.makedirs(save_dir, exist_ok=True)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Meta saved ->", meta_path)
print("MODEL:", MODEL, "| BASE_URL:", BASE_URL)
print(f"数据集是否有标注: {has_label_count > 0}")

In [ ]:
# Step 4: CAUSAL20-SC 因果分类推理代码
import json, os, re, time
import pandas as pd
from tqdm import tqdm
import requests

run_tag = f"{dataset_name}_{MODEL.replace('-', '_')}"
pred_path = os.path.join(save_dir, f"{run_tag}_predictions.csv")
err_path = os.path.join(save_dir, f"{run_tag}_errors.csv")

print(f"预测文件: {pred_path}")
print(f"错误文件: {err_path}")

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _make_causal_prompt(text: str):
    return (
        "Task: Classify whether the following sentence indicates a causal relationship between events.\n\n"
        f"Text: {text}\n\n"
        "Instructions:\n"
        "1. Analyze if the text describes a cause-and-effect relationship\n"
        "2. Return ONLY one word: either 'causal' or 'noise'\n"
        "3. 'causal' if the text indicates a causal relationship\n"
        "4. 'noise' if it does not indicate causality\n\n"
        "Your response:"
    )

def parse_causal_response(response_text: str):
    """解析模型响应，返回 'causal' 或 'noise'"""
    response_text = response_text.strip().lower()
    
    if 'causal' in response_text and 'noise' not in response_text:
        return 'causal'
    elif 'noise' in response_text:
        return 'noise'
    else:
        # 默认返回 noise
        return 'noise'

def ask_o3_mini_once(text):
    """调用o3-mini API"""
    url = f"{BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    user_text = _make_causal_prompt(text)
    
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a financial NLP expert. Respond only with 'causal' or 'noise'."},
            {"role": "user", "content": user_text}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        
        if response.status_code != 200:
            error_msg = f"API Error {response.status_code}: {response.text}"
            return None, "", False, error_msg
            
        result = response.json()
        
        if 'choices' not in result or len(result['choices']) == 0:
            return None, "", False, "No choices in response"
            
        txt = result['choices'][0]['message']['content']
        txt = _strip_code_fences(txt)
        pred_label = parse_causal_response(txt)
        return pred_label, txt, True, None
        
    except Exception as e:
        return None, "", False, str(e)

def ask_o3_mini(text):
    delay = 2.0
    for attempt in range(6):
        try:
            pred_label, raw_response, success, error = ask_o3_mini_once(text)
            if success:
                return pred_label, raw_response, True, None
            else:
                if attempt < 5:
                    time.sleep(delay)
                    delay = min(delay * 2, 60)
                    continue
                return None, raw_response, False, error
        except Exception as e:
            if attempt == 5:
                return None, "", False, str(e)
            time.sleep(delay)
            delay = min(delay * 2, 60)
    return None, "", False, "Exhausted retries"

# 先测试一个样本
print("测试单个样本...")
test_sample = ds[0]
test_text = test_sample["text"]
print(f"测试文本: {test_text[:100]}...")

test_pred, test_response, test_success, test_error = ask_o3_mini(test_text)
print(f"测试成功: {test_success}")
if test_success:
    print(f"预测标签: {test_pred}")
    print(f"真实标签: {test_sample['label']}")
    print(f"原始响应: {test_response[:100]}...")
    
    print("\n开始完整评估...")
    
    # 删除旧文件以重新运行
    if os.path.exists(pred_path):
        os.remove(pred_path)
    if os.path.exists(err_path):
        os.remove(err_path)
    
    err_rows = []
    buf = []
    save_every = 20

    total = len(ds)
    print(f"Starting o3-mini CAUSAL20-SC evaluation on {total} samples...")

    for i in tqdm(range(total)):
        x = ds[i]
        text = x["text"]
        gold_label = x["label"] if x["has_label"] else None

        try:
            pred_label, raw_response, success, error_msg = ask_o3_mini(text)
            if not success:
                raise RuntimeError(error_msg)
        except Exception as e:
            pred_label = "UNKNOWN"
            raw_response = f"ERROR: {type(e).__name__}: {e}"
            err_rows.append({"row_idx": i, "error": raw_response, "text": text})
            success = False

        buf.append({
            "row_idx": i,
            "text": text[:200] + "..." if len(text) > 200 else text,
            "pred_raw": raw_response,
            "pred_label": pred_label,
            "gold_label": gold_label,
            "has_label": x["has_label"],
            "success": success
        })

        if len(buf) % save_every == 0:
            out = pd.DataFrame(buf).sort_values("row_idx")
            out.to_csv(pred_path, index=False)
            if err_rows:
                pd.DataFrame(err_rows).to_csv(err_path, index=False)
            print(f"[checkpoint] saved {len(out)}/{total} -> {pred_path}")

    out = pd.DataFrame(buf).sort_values("row_idx")
    out.to_csv(pred_path, index=False)
    if err_rows:
        pd.DataFrame(err_rows).to_csv(err_path, index=False)
    print(f"[done] o3-mini CAUSAL20-SC evaluation completed -> {pred_path}")
    
else:
    print(f"测试失败: {test_error}")

In [ ]:
# Step 5: Compute evaluation metrics for CAUSAL20-SC
%pip install -q scikit-learn

import pandas as pd
import json
import time
from sklearn.metrics import accuracy_score, f1_score, classification_report

if os.path.exists(pred_path):
    df = pd.read_csv(pred_path).sort_values("row_idx").drop_duplicates("row_idx", keep="last")
    
    # 只考虑有标注的样本
    if 'has_label' in df.columns:
        labeled_df = df[df['has_label'] == True].copy()
    else:
        labeled_df = df[df['gold_label'] != 'unknown'].copy()
    
    print(f"总样本数: {len(df)}")
    print(f"有标注的样本数: {len(labeled_df)}")
    print(f"无标注的样本数: {len(df) - len(labeled_df)}")
    
    # 只考虑成功的预测
    if 'success' in labeled_df.columns:
        success_df = labeled_df[labeled_df['success'] == True].copy()
    else:
        success_df = labeled_df.copy()
    
    print(f"成功预测的有标注样本: {len(success_df)}")
    
    if len(success_df) > 0:
        y_true = success_df['gold_label'].tolist()
        y_pred = success_df['pred_label'].tolist()
        
        # 计算指标
        accuracy = accuracy_score(y_true, y_pred)
        f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
        f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        
        print("\n" + "="*50)
        print("EVALUATION RESULTS - o3-mini on FLARE-CAUSAL20-SC")
        print("="*50)
        print(f"Accuracy: {accuracy:.4f}")
        print(f"F1-Macro: {f1_macro:.4f}")
        print(f"F1-Weighted: {f1_weighted:.4f}")
        
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, zero_division=0))
        
        # 保存评估结果
        eval_results = {
            "model": MODEL,
            "dataset": "ChanceFocus/flare-causal20-sc",
            "split": "test",
            "total_samples": len(df),
            "labeled_samples": len(labeled_df),
            "successful_predictions": len(success_df),
            "accuracy": float(accuracy),
            "f1_macro": float(f1_macro),
            "f1_weighted": float(f1_weighted),
            "evaluation_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())
        }
        
        eval_path = os.path.join(save_dir, f"{run_tag}_evaluation_results.json")
        with open(eval_path, "w") as f:
            json.dump(eval_results, f, indent=2)
        print(f"\n评估结果已保存 -> {eval_path}")
        
    else:
        print("没有成功预测的有标注样本")
        
        # 如果完全没有标注，输出推理统计
        print("\n" + "="*50)
        print("INFERENCE RESULTS - o3-mini on FLARE-CAUSAL20-SC")
        print("="*50)
        print("注意: 测试集没有标注，只能进行推理测试")
        
        pred_counts = df['pred_label'].value_counts()
        print(f"\n预测分布:")
        for label, count in pred_counts.items():
            print(f"  {label}: {count} ({count/len(df):.2%})")
        
        eval_results = {
            "model": MODEL,
            "dataset": "ChanceFocus/flare-causal20-sc",
            "split": "test",
            "total_samples": len(df),
            "has_labels": False,
            "prediction_distribution": pred_counts.to_dict(),
            "note": "测试集无标注，仅进行推理测试",
            "evaluation_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())
        }
        
        eval_path = os.path.join(save_dir, f"{run_tag}_inference_results.json")
        with open(eval_path, "w") as f:
            json.dump(eval_results, f, indent=2)
        print(f"\n推理结果已保存 -> {eval_path}")
        
else:
    print(f"预测文件不存在: {pred_path}")